# STSM Evaluation of Saved Models — Threshold from Normal Training Data

No training. For each run and window, the saved best model is loaded and evaluated with **one change**:
the anomaly threshold is the `pc`-th percentile of the reconstruction errors of the **normal (label-0)
training pairs**, instead of all training pairs.

Everything else is as in training: same windows, same data split, same scaler (fitted on normal training
pairs), same seismic criterion and seismic-ratio calculation.

The results are saved in the same format as the training results, under a new model name
(`<MODEL_NAME>-NormalThr`), so `DEMETER_STSM-Results-Analysis.ipynb` can analyse them directly.

## Setup (PC or Google Colab)

In [1]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\Attention_Model-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/[FuSe][STSM] Megha Babu/Attention model/Attention_Model-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/Attention_Model-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

Project: D:\GIT\Attention_Model-Demeter-Seismic-Study


In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from lstm import SeismicCriteria, generate_windows, load_eq_catalog, prepare_window
from lstm.pipeline import process_split, run_seismic_analysis, plot_error_distributions
from lstm.STSM import RecurrentAutoencoder, model_name

## Settings

- `MODEL_NAME`: model whose saved models are evaluated (e.g. `LSTM-TA`).
- `MODELS_ROOT`: folder with the downloaded `best_model_Hp_tw48-30dBG_{run}-{MODEL_NAME}_w{i}.pth` files.
  They are searched in all its subfolders, so any layout works.
- Runs with a missing model file for any window are skipped.

In [3]:
# Data
DATA_DIR = PROJECT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"
EQ_PATH = DATA_DIR / "EQ.csv"

# Rolling windows (as in training)
train_months, val_months, stride = 12, 3, 3
WINDOW_START, WINDOW_END = "2005-01-01 00:00:00", "2011-01-02 00:00:00"
min_data_points = 17

# Anomaly threshold (percentile) and seismic criterion
pc = 98   # percentile of the NORMAL training errors
tw = 48   # time window (hours)
sw = 22   # spatial width (degrees)

# Models
MODEL_NAME = "LSTM-TA"
RUNS = list("ABCDE") + [f"In-{c}" for c in "ABCDE"]
MODELS_ROOT = PROJECT / "Result Files" / "STSM-Results" / MODEL_NAME

# Outputs
OUT_NAME = f"{MODEL_NAME}-NormalThr"
OUT_ROOT = PROJECT / "Result Files" / "STSM-Results" / OUT_NAME
RESUME = True   # skip windows already in the result CSV

def checkpoint(run, i):
    """Saved best model of a run and window, searched anywhere under MODELS_ROOT (None if missing)."""
    return next(MODELS_ROOT.rglob(f"best_model_Hp_tw{tw}-30dBG_{run}-{MODEL_NAME}_w{i}.pth"), None)

## Model loading and evaluation with the normal-data threshold

`load_model` rebuilds the autoencoder from the checkpoint itself (cell type, bidirectional, attention,
hidden size, layers, latent size), so runs trained with different settings are loaded correctly.

In [4]:
def load_model(path):
    """RecurrentAutoencoder with the architecture stored in the checkpoint."""
    state = torch.load(path, map_location="cpu")
    w = {k.replace(".lstm.", ".rnn."): v for k, v in state.items()}
    hh = w["encoder.rnn.weight_hh_l0"]
    hidden = hh.shape[1]
    model = RecurrentAutoencoder(
        input_size=w["encoder.rnn.weight_ih_l0"].shape[1],
        hidden_size=hidden,
        num_layers=sum(k.startswith("encoder.rnn.weight_ih_l") and not k.endswith("_reverse") for k in w),
        latent_dim=w["encoder.encoder_fc.weight"].shape[0],
        cell={4: "LSTM", 3: "GRU"}[hh.shape[0] // hidden],
        bidirectional=any(k.endswith("_reverse") for k in w),
        attention=("feature_temporal" if "encoder.feature_attention.weight" in w
                   else "temporal" if "encoder.temporal_attention.weight" in w else None),
    )
    model.load_state_dict(w)
    return model


def evaluate_window_normal_threshold(i, w, model, splits, datasets, scaled, eq, seismic_criteria, pc, m, output_dir):
    """
    As lstm.pipeline.evaluate_window, but the threshold is the pc-th percentile of the errors of the
    normal (label-0) training pairs. It is applied to all train and validation pairs.
    """
    normal = process_split(model, DataLoader(scaled["Train_normal"], batch_size=1), threshold_percentile=pc)
    threshold = normal["threshold_agg"]
    train = process_split(model, DataLoader(scaled["Train"], batch_size=1), threshold_agg=threshold)
    val = process_split(model, DataLoader(scaled["Val"], batch_size=1), threshold_agg=threshold)
    plot_error_distributions(i, m, train, val, pc, output_dir)

    rows = []
    for split, res in (("Val", val), ("Train", train)):
        sa = run_seismic_analysis(splits[split], datasets[split], eq, seismic_criteria, res["anomalies_agg"])
        rows.append({
            "window": i,
            "split": split,
            **w,
            "anomalies": len(res["anomalies_agg"]),
            "agg_value": sa["value"],
            "agg_error": sa["error"],
            "agg_total_eq": sa["total_eq"],
            "seismic_indices": sa["seismic_indices"],
            "threshold_agg": threshold,
            "threshold_source": "normal_train",
        })
    return rows

## Earthquake catalogue, rolling windows and available models

In [5]:
eq = load_eq_catalog(EQ_PATH)
seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
windows = generate_windows(WINDOW_START, WINDOW_END, train_months, val_months, stride)

runs = [run for run in RUNS if all(checkpoint(run, i) for i in range(len(windows)))]
print("Windows:", len(windows))
print("Runs with all models:", runs)
print("Runs skipped:        ", [run for run in RUNS if run not in runs])

Windows: 20
Runs with all models: ['A', 'B', 'C', 'D', 'E', 'In-D', 'In-E']
Runs skipped:         ['In-A', 'In-B', 'In-C']


## Evaluation loop

In [ ]:
for run in runs:
    m = f"tw{tw}-30dBG_{run}-{OUT_NAME}"
    output_dir = OUT_ROOT / m / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    results_csv = output_dir / f"result_Hp_{m}.csv"

    done = pd.read_csv(results_csv) if RESUME and results_csv.exists() else pd.DataFrame(columns=["window"])
    results = done.to_dict("records")
    print(f"\n{'#' * 90}\nRun {run}: {m}\n{'#' * 90}")

    for i, w in enumerate(windows):
        if i in set(done["window"]):
            continue
        print(f"Window {i}: Val {w['val_start'].date()} -> {w['val_end'].date()}")
        splits, datasets, scaled = prepare_window(
            BG_WINDOW_DIR / f"Background_data-window_{i}.pkl",
            {s: LABEL_DIR / f"summary_df_{s.lower()}_30D-{sw}SW-tw{tw}_w{i}.csv" for s in ("Train", "Val")},
            w,
            min_data_points,
        )
        model = load_model(checkpoint(run, i))
        results += evaluate_window_normal_threshold(i, w, model, splits, datasets, scaled, eq, seismic_criteria,
                                                    pc, m, output_dir)
        pd.DataFrame(results).to_csv(results_csv, index=False)  # saved after every window

print("\nResults in:", OUT_ROOT)


##########################################################################################
Run A: tw48-30dBG_A-LSTM-TA-NormalThr
##########################################################################################
Window 0: Val 2006-01-01 -> 2006-04-01
Pairs: Train_normal=742, Val_normal=215, Train=1553, Val=381
[INFO] Total valid combined sequences: 381
[INFO] Total valid combined sequences: 1553
Window 1: Val 2006-04-01 -> 2006-07-01
Pairs: Train_normal=781, Val_normal=207, Train=1515, Val=434
[INFO] Total valid combined sequences: 434
[INFO] Total valid combined sequences: 1515
Window 2: Val 2006-07-01 -> 2006-10-01
Pairs: Train_normal=802, Val_normal=211, Train=1572, Val=451
[INFO] Total valid combined sequences: 451
[INFO] Total valid combined sequences: 1572
Window 3: Val 2006-10-01 -> 2007-01-01
Pairs: Train_normal=825, Val_normal=188, Train=1629, Val=408
[INFO] Total valid combined sequences: 408
[INFO] Total valid combined sequences: 1629
Window 4: Val 2007-01-01 -> 200

## Next step

Open `DEMETER_STSM-Results-Analysis.ipynb` and set `MODEL_NAME = "LSTM-TA-NormalThr"` (i.e. `OUT_NAME`)
to get the Z-factor, average ratio and baseline comparison of these results.